In [3]:
import os
import csv
import pandas as pd
import re
import subprocess
import numpy as np
import scipy.stats as stats

new_dir = "/home/jingqi"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi'

## Cluster

In [28]:
import pandas as pd
import os

output_dir = "RNALocateV3.0/Data/MEME_New/Clusters_Top"
os.makedirs(output_dir, exist_ok=True)

prob_df = pd.read_csv("RNALocateV3.0/Data/Main/Probabilities.csv")
id_col = prob_df.columns[0]
prob_df = prob_df.set_index(id_col)

fasta_dict = {}
with open("RNALocateV3.0/Data/Raw/3UTR_clean.txt", "r") as f:
    seq_id = ""
    seq = []
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            if seq_id:
                fasta_dict[seq_id] = "".join(seq)
            seq_id = line[1:]
            seq = []
        else:
            seq.append(line)
    if seq_id:
        fasta_dict[seq_id] = "".join(seq)

# Pre-map CSV IDs to FASTA headers to guarantee sequence availability
valid_ids = {}
for pid in prob_df.index:
    for header in fasta_dict.keys():
        if str(pid) in header:
            valid_ids[pid] = header
            break

# Restrict the probability matrix to ONLY transcripts with a physical 3' UTR
prob_df_filtered = prob_df.loc[list(valid_ids.keys())]
compartments = prob_df_filtered.select_dtypes(include='number').columns

for compartment in compartments:
    pos_file = os.path.join(output_dir, f"{compartment.replace(' ', '_')}_top200.fasta")
    
    # Sort the validated subset and apply the threshold
    sorted_df = prob_df_filtered.sort_values(by=compartment, ascending=False)
    valid_hits = sorted_df[sorted_df[compartment] > 0.5]
    
    # Extract exactly the top 200 available
    top_200_ids = valid_hits.head(150).index
    
    pos_count = 0
    with open(pos_file, "w") as pos_out:
        for pid in top_200_ids:
            matched_header = valid_ids[pid]
            sequence = fasta_dict[matched_header]
            pos_out.write(f">{matched_header}\n{sequence}\n")
            pos_count += 1
                
    print(f"{compartment}: Extracted {pos_count} top-confidence sequences for MEME.")

chromatin_prob: Extracted 150 top-confidence sequences for MEME.
cytoplasm_prob: Extracted 150 top-confidence sequences for MEME.
cytosol_prob: Extracted 150 top-confidence sequences for MEME.
endoplasmic reticulum_prob: Extracted 69 top-confidence sequences for MEME.
extracellular region_prob: Extracted 150 top-confidence sequences for MEME.
membrane_prob: Extracted 150 top-confidence sequences for MEME.
mitochondrion_prob: Extracted 150 top-confidence sequences for MEME.
nucleolus_prob: Extracted 150 top-confidence sequences for MEME.
nucleoplasm_prob: Extracted 150 top-confidence sequences for MEME.
nucleus_prob: Extracted 150 top-confidence sequences for MEME.
ribosome_prob: Extracted 150 top-confidence sequences for MEME.


## Parameters

In [22]:
import subprocess
from Bio import motifs
from Bio.motifs import meme
import re

# ===== MOTIFS ELICITATION CONFIGURATION =====
# label_names = [
#     "chromatin", "cytoplasm", "cytosol", "ER", "extracellular", 
#     "membrane", "mitochondrion", "nucleolus", "nucleoplasm", "nucleus", "ribosome"
# ]


# Paths
meme_path = "miniconda3/envs/rnalocate_old/bin/meme"
tomtom_path = "miniconda3/envs/rnalocate_old/bin/tomtom"
markov_path = "miniconda3/envs/rnalocate_old/bin/fasta-get-markov"
input_dir = "RNALocateV3.0/Data/MEME_New/Clusters_Top"
meme_output_dir = "RNALocateV3.0/Data/MEME_New/meme_results"
tomtom_output_dir = "RNALocateV3.0/Data/TOMTOM_New/tomtom_results"

# Databases
databases = {
    "CISBP_RNA": "~/meme_db/motif_databases/CISBP-RNA/Mus_musculus.dna_encoded.meme",
}


# Create output directories
os.makedirs(meme_output_dir, exist_ok=True)
os.makedirs(tomtom_output_dir, exist_ok=True)
for db_name in databases.keys():
    os.makedirs(f"{tomtom_output_dir}/{db_name}", exist_ok=True)

# count sequences in FASTA file
def count_fasta_sequences(fasta_path):
    count = 0
    with open(fasta_path, 'r') as f:
        for line in f:
            if line.startswith('>'):
                count += 1
    return count

print(" Configuration loaded")


# ===== FILTERING CONFIGRATION ===== 
MIN_SITE_PERCENT = 0.02
E_VALUE_THRESH = 0.1 
P_VALUE_THRESH = 0.0005
Q_VALUE_THRESH = 0.1

# Parse MEME results to get site counts per motif
def get_motif_sites(meme_txt_file):
    """Extract number of sites for each motif from MEME output."""
    motif_sites = {}
    
    with open(meme_txt_file, 'r') as f:
        content = f.read()
    
    import re
    pattern = r'MOTIF\s+(\S+).*?sites\s*=\s*(\d+)'
    matches = re.findall(pattern, content, re.DOTALL)
    
    for motif_id, sites in matches:
        motif_sites[motif_id] = int(sites)
    
    return motif_sites
    
print(" ALL SET!")

 Configuration loaded
 ALL SET!


## MEME

In [29]:
# some interruption happended. now rerun from the middle
label_names = [
    "extracellular", "membrane", "mitochondrion", "nucleolus", "nucleoplasm", "nucleus", "ribosome",
    "chromatin", "cytoplasm", "cytosol", "ER"
]


# Generate the global background model
background_fasta = "RNALocateV3.0/Data/Raw/3UTR_clean.txt"
bfile_path = "RNALocateV3.0/Data/MEME_New/background_model.bfile"

print("Calculating 0-order Markov background model...")
subprocess.run([markov_path, "-m", "0", background_fasta, bfile_path], check=True)

file_mapping = {
    "chromatin": "chromatin_prob",
    "cytoplasm": "cytoplasm_prob",
    "cytosol": "cytosol_prob",
    "ER": "endoplasmic_reticulum_prob",
    "extracellular": "extracellular_region_prob",
    "membrane": "membrane_prob",
    "mitochondrion": "mitochondrion_prob",
    "nucleolus": "nucleolus_prob",
    "nucleoplasm": "nucleoplasm_prob",
    "nucleus": "nucleus_prob",
    "ribosome": "ribosome_prob"
}

for label in label_names:
    # Adjust this filename pattern to exactly match your clustering output
    file_prefix = file_mapping[label]
    fasta_file = f"{input_dir}/{file_prefix}_top200.fasta"
    output_folder = f"{meme_output_dir}/{label}_positive"
    
    if not os.path.exists(fasta_file):
        print(f"File missing for {label}, skipping...")
        continue
        
    meme_cmd = [
        meme_path, fasta_file,
        "-dna",
        "-oc", output_folder,
        "-mod", "zoops",
        "-nmotifs", "10",
        "-minw", "4",
        "-maxw", "10",
        "-bfile", bfile_path,
        "-maxsize", "1000000"
    ]
    
    print(f"Running MEME for {label}...")
    subprocess.run(meme_cmd, check=True)

print("MEME motif discovery complete.")

Calculating 0-order Markov background model...
2432 1 11355 1234.1 3001379
Running MEME for extracellular...


The output directory 'RNALocateV3.0/Data/MEME_New/meme_results/extracellular_positive' already exists.
Its contents will be overwritten.
Initializing the motif probability tables for 2 to 150 sites...
nsites = 150
Done initializing.
SEEDS: highwater mark: seq 149 pos 201

seqs=   150, min=  16, max= 8742, total=   171869

motif=1
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=2
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=3
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=4
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=5
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=6
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=7
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=8
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=9
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=10
SEED WIDTHS: 4 6 8 10
em: w=  10, psi

Running MEME for membrane...


Writing results to output directory 'RNALocateV3.0/Data/MEME_New/meme_results/membrane_positive'.
Initializing the motif probability tables for 2 to 150 sites...
nsites = 150
Done initializing.
SEEDS: highwater mark: seq 149 pos 3167

seqs=   150, min=  86, max= 5223, total=   213804

motif=1
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=2
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=3
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=4
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=5
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=6
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=7
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=8
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=9
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=10
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     


Running MEME for mitochondrion...


...

Running MEME for cytosol...


The output directory 'RNALocateV3.0/Data/MEME_New/meme_results/cytosol_positive' already exists.
Its contents will be overwritten.
Initializing the motif probability tables for 2 to 150 sites...
nsites = 150
Done initializing.
SEEDS: highwater mark: seq 149 pos 1287

seqs=   150, min=  12, max= 8742, total=   229801

motif=1
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=2
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=3
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=4
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=5
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=6
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=7
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=8
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=9
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 150, iter=  40     
motif=10
SEED WIDTHS: 4 6 8 10
em: w=  10, psites= 

Running MEME for ER...


The output directory 'RNALocateV3.0/Data/MEME_New/meme_results/ER_positive' already exists.
Its contents will be overwritten.
Initializing the motif probability tables for 2 to 69 sites...
nsites = 69
Done initializing.
SEEDS: highwater mark: seq 68 pos 2938

seqs=    69, min=  97, max= 6337, total=   110651

motif=1
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=2
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=3
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=4
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=5
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=6
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=7
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=8
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=9
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40    
motif=10
SEED WIDTHS: 4 6 8 10
em: w=  10, psites=  69, iter=  40   

MEME motif discovery complete.


## TOMTOM

In [30]:
for class_name in label_names:
    for bucket in ['positive', 'negative']:
        meme_results = f"{meme_output_dir}/{class_name}_{bucket}/meme.txt"
        
        if not os.path.exists(meme_results):
            continue
        
        print(f" TOMTOM: {class_name} {bucket}")
        
        for db_name, db_path in databases.items():
            db_path_expanded = os.path.expanduser(db_path)
            
            if not os.path.exists(db_path_expanded):
                print(f"  Database not found: {db_name}")
                continue
            
            tomtom_out = f"{tomtom_output_dir}/{db_name}/{class_name}_{bucket}"
            
            tomtom_cmd = [
                tomtom_path,
                "-oc", tomtom_out,
                "-thresh", "1.0",  # Permissive, filter later
                "-evalue",
                "-no-ssc",
                meme_results,
                db_path_expanded
            ]
            
            try:
                subprocess.run(tomtom_cmd, check=True, capture_output=True)
            except subprocess.CalledProcessError as e:
                print(f"  Error in {db_name}: {e.stderr.decode()[:100]}")

print("\n TOMTOM complete")


 TOMTOM: extracellular positive
 TOMTOM: membrane positive
 TOMTOM: mitochondrion positive
 TOMTOM: nucleolus positive
 TOMTOM: nucleoplasm positive
 TOMTOM: nucleus positive
 TOMTOM: ribosome positive
 TOMTOM: chromatin positive
 TOMTOM: cytoplasm positive
 TOMTOM: cytosol positive
 TOMTOM: ER positive

 TOMTOM complete


## Filter

In [35]:
label_names

['extracellular',
 'membrane',
 'mitochondrion',
 'nucleolus',
 'nucleoplasm',
 'nucleus',
 'ribosome',
 'chromatin',
 'cytoplasm',
 'cytosol',
 'ER']

In [46]:
# ===== FILTERING CONFIGRATION ===== 
MIN_SITE_PERCENT = 0.15
E_VALUE_THRESH = 0.1 
P_VALUE_THRESH = 0.0005
Q_VALUE_THRESH = 0.05
total_seqs = 150

# Pre-load the CISBP-RNA database names
cisbp_db_path = os.path.expanduser("~/meme_db/motif_databases/CISBP-RNA/Mus_musculus.dna_encoded.meme")
id_to_name = {}

try:
    with open(cisbp_db_path, 'r') as f:
        for line in f:
            if line.startswith("MOTIF"):
                # Example line: MOTIF M043_0.6 Pcbp2
                parts = line.strip().split()
                if len(parts) >= 3:
                    motif_id = parts[1]
                    protein_name = parts[2]
                    id_to_name[motif_id] = protein_name
    print(f"Loaded {len(id_to_name)} biological names from database.")
except FileNotFoundError:
    print(f"Warning: Could not find database at {cisbp_db_path}. Protein names will be blank.")

filtered_results = []

for class_name in label_names:
    for bucket in ['positive', 'negative']:
        
        # Setup Input Paths
        # Ensure file_prefix is defined correctly in your broader script
        file_prefix = file_mapping.get(class_name, class_name) 
        fasta_file = f"{input_dir}/{file_prefix}_top200.fasta"
        meme_file = f"{meme_output_dir}/{class_name}_{bucket}/meme.txt"
        
        if not os.path.exists(fasta_file) or not os.path.exists(meme_file):
            continue
        
        # Get MEME Sites
        motif_sites = {}
        with open(meme_file, 'r') as f:
            content = f.read()
            matches = re.findall(r'MOTIF\s+(\S+).*?sites\s*=\s*(\d+)', content, re.DOTALL)
            for m_id, m_sites in matches:
                # Filter out junk
                if m_id.isdigit():
                    motif_sites[m_id] = int(m_sites)

        for db_name in databases.keys():
            txt_file = f"{tomtom_output_dir}/{db_name}/{class_name}_{bucket}/tomtom.txt"
            
            if not os.path.exists(txt_file):
                continue
            
            try:
                # Read without comment='#' so to keep the header row
                df = pd.read_csv(txt_file, sep='\t')
                
                # Sanitize column names:
                df.columns = df.columns.str.replace('#', '', regex=False).str.strip().str.replace(' ', '_')
                
            except Exception as e:
                print(f"Error reading {txt_file}: {e}")
                continue
            
            if df.empty or 'Query_ID' not in df.columns:
                continue

            # Filter by rows
            for _, row in df.iterrows():
                    
                # Filter 0: Direction
                direction = row.get('Orientation','')
                if direction == '-':
                    continue
                
                # Filter 1: Site Count
                query_motif = str(row['Query_ID'])
                sites = motif_sites.get(query_motif, 0)
                ratio = float(sites / total_seqs)
                if ratio < MIN_SITE_PERCENT:
                    continue
                
                # Filter 2: Statistical Thresholds
                try:
                    e_val = float(row.get('E-value', 1.0))
                    p_val = float(row.get('p-value', 1.0))
                    q_val = float(row.get('q-value', 1.0))
                except ValueError:
                    continue
                
                passes = sum([
                    e_val < E_VALUE_THRESH,
                    p_val < P_VALUE_THRESH,
                    q_val < Q_VALUE_THRESH
                ])
                
                if passes >= 1:
                    target_id = row.get('Target_ID', '')
                    # Map the target ID to the biological name
                    protein_name = id_to_name.get(target_id, "Unknown")
                    
                    filtered_results.append({
                        'Class': class_name,
                        'Bucket': bucket,
                        'Database': db_name,
                        'Query_motif': query_motif,
                        'Target': target_id,
                        'Protein_Name': protein_name,
                        'Abandance': f'{ratio*100:.2f}%',
                        'E-value': e_val,
                        'p-value': p_val,
                        'q-value': q_val,
                        'Overlap': row.get('Overlap', '')
                    })

# Save results
df_filtered = pd.DataFrame(filtered_results)
output_file = "RNALocateV3.0/Data/TOMTOM_New/RBP_filtered.csv"
# Ensure the directory exists before saving
os.makedirs(os.path.dirname(output_file), exist_ok=True)
df_filtered.to_csv(output_file, index=False)

print(f"\nFILTERING SUMMARY")
print(f"Total matches passing filters: {len(df_filtered)}")

if not df_filtered.empty:
    print(f"\nTop 10 matches:")
    print(df_filtered.head(10).to_string())

Loaded 93 biological names from database.

FILTERING SUMMARY
Total matches passing filters: 89

Top 10 matches:
           Class    Bucket   Database Query_motif    Target                         Protein_Name Abandance   E-value   p-value   q-value  Overlap
0  extracellular  positive  CISBP_RNA           1  M043_0.6    (Pcbp2)_(Homo_sapiens)_(RBD_1.00)   100.00%  0.065558  0.000705  0.127317        7
1  extracellular  positive  CISBP_RNA           3  M328_0.6   (Elavl2)_(Homo_sapiens)_(RBD_1.00)    70.67%  0.052453  0.000564  0.098919       10
2       membrane  positive  CISBP_RNA           2  M043_0.6    (Pcbp2)_(Homo_sapiens)_(RBD_1.00)    88.67%  0.045893  0.000493  0.044060        7
3       membrane  positive  CISBP_RNA           4  M042_0.6  (Gm10110)_(Homo_sapiens)_(RBD_0.97)    74.67%  0.047856  0.000515  0.090378        7
4       membrane  positive  CISBP_RNA           6  M157_0.6    (Celf3)_(Homo_sapiens)_(RBD_0.83)    39.33%  0.021212  0.000228  0.020745        7
5       memb

In [48]:
import pandas as pd

df = pd.read_csv(output_file)

grouped = df.groupby('Target')['Class'].unique().apply(list)

expanded_df = pd.DataFrame(grouped.tolist(), index=grouped.index)

expanded_df.columns = [f'Class {i+1}' for i in range(expanded_df.shape[1])]

expanded_df.reset_index(inplace=True)

# Generate a 1:1 mapping from the original dataframe
target_to_protein = dict(zip(df['Target'], df['Protein_Name']))

# Insert Protein_Name exactly at position 1 (the second column)
expanded_df.insert(1, 'Protein_Name', expanded_df['Target'].map(target_to_protein))

expanded_df.fillna('', inplace=True)

# Export to a new CSV file
check_file = '/home/jingqi/RNALocateV3.0/Data/TOMTOM_New/Target_Class_Matrix.csv'
expanded_df.to_csv(check_file, index=False)

print(f"Matrix successfully generated and saved to {check_file}")
print(f"Total unique targets: {len(expanded_df)}")

Matrix successfully generated and saved to /home/jingqi/RNALocateV3.0/Data/TOMTOM_New/Target_Class_Matrix.csv
Total unique targets: 17
